### Importing the required libraries

In [1]:
import pandas as pd 
import glob
import requests 
import sqlite3
import numpy as np
from datetime import datetime 
from bs4 import BeautifulSoup

url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'
html_page = requests.get(url).text
data = BeautifulSoup(html_page, 'html.parser')

log_file = 'etl_project_log.txt'
csv_path = 'Countries_by_GDP.csv'
db_name = 'World_Economics.db'
table_name = 'Countries_by_GDP'
table_attribs = ['Country', 'GDP_USD_billion']

### Task1: Extracting Information

In [13]:
def extract(url, table_attribs):
    page = requests.get(url).text
    data = BeautifulSoup(page,'html.parser')
    df = pd.DataFrame(columns=table_attribs)
    tables = data.find_all('tbody')
    rows = tables[2].find_all('tr')
    
    for row in rows:
        col = row.find_all('td')
        if len(col)!=0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {"Country": col[0].a.contents[0],
                             "GDP_USD_millions": col[2].contents[0]}
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df,df1], ignore_index=True)
    return df

### Task2: Transform Information

In [14]:
def transform(df):
    GDP_list = df["GDP_USD_millions"].tolist()
    GDP_list = [float("".join(x.split(','))) for x in GDP_list]
    GDP_list = [np.round(x/1000,2) for x in GDP_list]
    df["GDP_USD_millions"] = GDP_list
    df=df.rename(columns = {"GDP_USD_millions":"GDP_USD_billions"})
    return df

### Task3: Loading Data to CSV, DB

In [15]:
def load_to_csv(df, csv_path):
    df.to_csv(csv_path)

def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

### Task4: QUerying Database Table


In [16]:
def run_query(query_statement, sql_connection):
    print(query_statement)
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)

### Task5: Logging Progress

In [17]:
def log_progress(message):
    timestamp_format = '%Y-%h-%d-%H:%M:%S' 
    now = datetime.now()
    timestamp = now.strftime(timestamp_format)
    with open('./etl_project_log.txt', 'a') as f:
        f.write(timestamp + " : " + message + '\n')

### Function Calls


In [19]:
log_progress("Preliminaries complete. Initialing ETL Process.")
df =extract(url, table_attribs)

log_progress('Data extraction complete. Initialing Transformation process.')
df = transform(df)

log_progress('Data transformation complete. Initialing loading process.')
load_to_csv(df, csv_path)
log_progress('Data saved to CSV file.')

sql_connection = sqlite3.connect(db_name)
log_progress('SQL Connection Initiated.')
load_to_db(df, sql_connection, table_name)
log_progress("Data loaded to Database as table. Running the query.")

query_statement = f"SELECT * FROM {table_name} WHERE GDP_USD_billions >= 100"
run_query(query_statement, sql_connection)

log_progress('Process Complete.')

sql_connection.close()

SELECT * FROM Countries_by_GDP WHERE GDP_USD_billions >= 100
          Country GDP_USD_billion  GDP_USD_billions
0   United States            None          26854.60
1           China            None          19373.59
2           Japan            None           4409.74
3         Germany            None           4308.85
4           India            None           3736.88
..            ...             ...               ...
64          Kenya            None            118.13
65         Angola            None            117.88
66           Oman            None            104.90
67      Guatemala            None            102.31
68       Bulgaria            None            100.64

[69 rows x 3 columns]
